In [39]:
"""Compare manual liver tumors with selected segmentation models.

Notebook usage (``scans`` and ``xnat_project_path`` are existing variables)::

    from liver_tumor_segmentation_analysis import analyze_segmentations
    models = ['nnunet', 'totalseg']
    results = analyze_segmentations(scans, xnat_project_path, dice_root, models)

Helpers accept an experiment record with the original scan fields plus ``path``
(the experiment directory). Label None selects all nonzero voxels. Volumes use
header voxel sizes, assumed to be in mm. Masks must share a voxel grid;
both array shapes and NIfTI affines are checked before scoring.
"""
import os,sys,json,datetime,yaml,importlib,csv,nibabel as nib,numpy as np
# nibabel 5.4.x currently pulls NumPy 2.x, which can break this notebook stack.
! pip install nibabel==5.1.0
from pathlib import Path

# Library with XNAT Jupyter workflow Python and shell scripts.
pymipl_path = os.path.abspath('../')
sys.path.append(pymipl_path)
sys.path.append(os.path.abspath(pymipl_path + '/xnat_workflow'))

# dicom_sort is part of pymipl. It analyzes XNAT projects for structural scans
# and segmentations.
from dicom_sort import *

def dice_coefficient(mask_a, mask_b, label_a=None, label_b=None, liver_mask=None):
    """Compute Dice, optionally restricting both masks to positive liver voxels."""
    mask_a = np.asarray(mask_a, dtype=bool) if label_a is None else np.asarray(mask_a) == label_a
    mask_b = np.asarray(mask_b, dtype=bool) if label_b is None else np.asarray(mask_b) == label_b
    if mask_a.shape != mask_b.shape:
        raise ValueError(f'Mask shape mismatch: {mask_a.shape} versus {mask_b.shape}')
    if liver_mask is not None:
        liver_mask = np.asarray(liver_mask) > 0
        if liver_mask.shape != mask_a.shape:
            raise ValueError(f'Liver mask shape mismatch: {liver_mask.shape} versus {mask_a.shape}')
        mask_a = mask_a & liver_mask
        mask_b = mask_b & liver_mask
    denom = mask_a.sum() + mask_b.sum()
    return float(2 * np.logical_and(mask_a, mask_b).sum() / denom) if denom else 1.0


def convert_manual_seg_to_nifti(experiment, output_nifti):
    """Combine manual Tumor segments into one NIfTI; return segment count."""
    if __package__:
        from .test_highdicom import dicomseg2nii
    else:
        from test_highdicom import dicomseg2nii

    exp_dir, output_nifti = Path(experiment['path']), Path(output_nifti)
    manual_seg_dir = exp_dir / 'SCANS' / str(experiment['SegScan1']) / 'secondary'
    dicomseg_files = sorted(manual_seg_dir.glob('*.dcm'))
    if len(dicomseg_files) != 1:
        raise ValueError(f'Expected one DICOM SEG file in {manual_seg_dir}, found {len(dicomseg_files)}')
    reference_file = exp_dir / 'RESOURCES' / 'Colon-Mets-Liver-Segmentation-Toolbox-20260430' / 'ct_struct.nii.gz'
    if not reference_file.is_file():
        raise FileNotFoundError(reference_file)

    output_nifti.parent.mkdir(parents=True, exist_ok=True)
    component_files = dicomseg2nii(str(dicomseg_files[0]), str(reference_file), str(output_nifti.parent))
    tumor_files = [Path(p) for p in component_files if 'Tumor' in Path(p).name]
    if not tumor_files:
        raise ValueError(f'No Tumor segment masks written for {dicomseg_files[0]}')
    reference = nib.load(reference_file)
    manual_mask = np.zeros(reference.shape, dtype=bool)
    for filename in tumor_files:
        component = nib.load(filename).get_fdata() > 0
        if component.shape != manual_mask.shape:
            raise ValueError(f'Manual segment shape mismatch: {filename}')
        manual_mask |= component
    nib.save(nib.Nifti1Image(manual_mask.astype(np.uint8), reference.affine, reference.header), output_nifti)
    return len(tumor_files)


def get_model_segmentation_paths(experiment, models):
    """Return existing NIfTI paths and label indices in model-list order.
    Add model-specific paths and labels only to the mapping below.
    """
    model_specs = {
        'nnunet': ('nnunet_output/ct_struct.nii.gz', 9),
        'totalseg': ('totalseg_liver_lesions/liver_lesions.nii.gz', None),
        'difftumor': ('difftumor_liver_tumor.nii.gz',2)
    }

    model_dir = Path(experiment['path']) / 'RESOURCES' / 'ABDOMINAL_CT_SEGMENTATION' / ('scan' + str(experiment['StructScan']))
    paths, labels = [], []
    for model in models:
        if model not in model_specs:
            raise KeyError(f'Unknown segmentation model: {model}; available: {list(model_specs)}')
        relative_path, label = model_specs[model]
        path = model_dir / relative_path
        if not path.is_file():
            raise FileNotFoundError(f'{model}: {path}')
        paths.append(path)
        labels.append(label)
    return paths, labels


def get_liver_segmentation_path(experiment, model):
    """Return an existing liver NIfTI path for the selected liver model.

    Add liver model paths to liver_models; all positive labels form the liver.
    DiffTumor includes both liver tissue and tumor labels.
    """
    liver_models = {
        'gevaert': Path('Colon-Mets-Liver-Segmentation-Toolbox-20260430') / 'liver_mask.nii.gz',
        'difftumor': Path('ABDOMINAL_CT_SEGMENTATION') / ('scan' + str(experiment['StructScan'])) / 'difftumor_liver_tumor.nii.gz',
    }
    if model not in liver_models:
        raise KeyError(f'Unknown liver model: {model}; available: {list(liver_models)}')
    path = Path(experiment['path']) / 'RESOURCES' / liver_models[model]
    if not path.is_file():
        raise FileNotFoundError(f'{model}: {path}')
    return path


def segmentation_on_reference(image, reference):
    """Read labels after checking shape and physical voxel coordinates."""
    if len(image.shape) != 3 or len(reference.shape) != 3:
        raise ValueError('Segmentation images must be 3D')
    if image.shape != reference.shape or not np.allclose(image.affine, reference.affine, rtol=0, atol=1e-5):
        raise ValueError(f'Segmentation grid mismatch: {image.get_filename()} has shape {image.shape}, affine {image.affine}; reference has shape {reference.shape}, affine {reference.affine}. Resample the segmentation to the reference grid before scoring.')
    return image.get_fdata()


def tumor_volume_mm3(nifti_file, label=None):
    """Return selected tumor voxel count times header voxel volume in mm3."""
    image = nib.load(nifti_file)
    data = image.get_fdata()
    mask = data.astype(bool) if label is None else data == label
    return float(np.count_nonzero(mask) * np.prod(image.header.get_zooms()[:3]))


def write_to_csv(csv_file, models, processed_experiments, dice_scores, tumor_segments, tumor_volumes):
    """Write one row per experiment, with columns following model-list order."""
    volume_names = ['manual', *models]
    columns = [processed_experiments, *(dice_scores[m] for m in models), tumor_segments, *(tumor_volumes[m] for m in volume_names)]
    if len({len(column) for column in columns}) != 1:
        raise ValueError('Result lists have different lengths')
    csv_file = Path(csv_file)
    csv_file.parent.mkdir(parents=True, exist_ok=True)
    with csv_file.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['experiment', *(f'dice-score-{m}' for m in models), 'tumor-segments', *(f'tumor-volume-mm3-{m}' for m in volume_names)])
        writer.writerows(zip(*columns, strict=True))


def analyze_segmentations(scans, xnat_project_path, dice_root, models, stop_after_1=True, liver_model=None, diagnostics=False):
    """Collect per-experiment results and write segmentation_metrics_YYYYMMDD_HHMMSS.csv.

    Missing or invalid inputs skip the whole experiment with an error message.
    Segment counts refer to manual Tumor masks, not connected components.
    liver_model selects a shared liver mask for Dice; None uses all voxels.
    diagnostics prints geometry, label counts, and Dice with/without liver masking.
    """
    models = list(models)
    if not models or len(set(models)) != len(models) or 'manual' in models:
        raise ValueError('Provide a nonempty list of unique model names excluding manual')
    results = dict(processed_experiments=[], dice_scores={m: [] for m in models}, tumor_segments=[], tumor_volumes={m: [] for m in ['manual', *models]})
    i=0
    for scan in scans:
        experiment = dict(scan, path=Path(xnat_project_path) / scan['Experiment'])
        manual_file = Path(dice_root) / scan['Experiment'] / 'All_Tumors.nii.gz'
        try:
            paths, labels = get_model_segmentation_paths(experiment, models)
            segments = convert_manual_seg_to_nifti(experiment, manual_file)
            manual_image = nib.load(manual_file)
            manual_mask = manual_image.get_fdata() > 0
            if diagnostics:
                print(f"{scan['Experiment']} manual: {manual_file}; shape={manual_image.shape}; voxels={np.count_nonzero(manual_mask)}")
            liver_mask = None
            if liver_model is not None:
                liver_path = get_liver_segmentation_path(experiment, liver_model)
                liver_image = nib.load(liver_path)
                liver_mask = segmentation_on_reference(liver_image, manual_image) > 0
                if diagnostics:
                    print(f'Liver: {liver_path}; shape={liver_image.shape}')
                    print(f'Manual tumor voxels inside liver: {np.count_nonzero(manual_mask & liver_mask)}/{np.count_nonzero(manual_mask)}')
            scores = []
            for model, path, label in zip(models, paths, labels, strict=True):
                model_image = nib.load(path)
                model_data = segmentation_on_reference(model_image, manual_image)
                score = dice_coefficient(manual_mask, model_data, label_b=label, liver_mask=liver_mask)
                scores.append(score)
                if diagnostics:
                    values, counts = np.unique(model_image.get_fdata(), return_counts=True)
                    print(f'{model}: {path}; selected label={label}; shape={model_image.shape}')
                    print('Native label voxel counts:', dict(zip(values.tolist(), counts.tolist())))
                    model_mask = model_data.astype(bool) if label is None else model_data == label
                    roi = np.ones(manual_mask.shape, dtype=bool) if liver_mask is None else liver_mask
                    print(f'Tumor voxels on manual grid: {np.count_nonzero(model_mask)}; inside liver: {np.count_nonzero(model_mask & roi)}; intersection: {np.count_nonzero(manual_mask & model_mask & roi)}')
                    full_dice = dice_coefficient(manual_mask, model_data, label_b=label)
                    print(f'Dice: full image={full_dice:.6f}, within liver={score:.6f}')
            volumes = [tumor_volume_mm3(manual_file), *(tumor_volume_mm3(p, label) for p, label in zip(paths, labels, strict=True))]
        except Exception as e:
            print(f"Skipping experiment {scan['Experiment']}: {e}")
            continue
            
        results['processed_experiments'].append(scan['Experiment'])
        results['tumor_segments'].append(segments)
        for model, score in zip(models, scores, strict=True):
            results['dice_scores'][model].append(score)
        for model, volume in zip(['manual', *models], volumes, strict=True):
            results['tumor_volumes'][model].append(volume)
        i=i+1
        if not i % 10: 
            print(f'analyzed {i}/{len(scans)} entries')
        if stop_after_1: break
            
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    write_to_csv(Path(dice_root) / f'segmentation_metrics_{timestamp}.csv', models, **results)
    return results


In [40]:
# XNAT project label
project='CRLM'

# Persistent workspace root path
root_dir=Path("/workspace/dnaxuser-wustl_mmilchenko_adapt")

local_workdir_path = root_dir / project
xnat_project_path = f'/data/projects/{project}/experiments'
directory_structure_file = local_workdir_path / "project_dir_structure.json"
xnat_structure_file = local_workdir_path / "xnat_structure.json"
scanlist_file = local_workdir_path / "scans.csv"

with open(directory_structure_file, 'r') as file:
    d = json.load(file)

subjects, scans = reindex_to_structurals_and_segs(
    d,
    xnat_structure_file,
    scanlist_file
)
print(f'Number of structural scans: {len(scans)}')
print('First scan: ', scans[0])

Number of structural scans: 197
First scan:  {'Subject': 'CRLM-CT-1087', 'Experiment': '12-10-1991-NA-CT_CHABDPEL_W_CONTRAST-52410', 'StructScan': '2', 'StructScanSerDesc': '', 'SegScan1': '100', 'SegScan1_SerDesc': 'Segmentation', 'SegScan1_SOPClass': 'Seg'}


In [ ]:
dice_root = Path('/workspace/dnaxuser-wustl_mmilchenko_adapt') / project / 'DICE_COMPUTE'
models=['nnunet','totalseg','difftumor']
analyze_segmentations(scans,xnat_project_path,dice_root,models,stop_after_1=False,liver_model='gevaert',diagnostics=True)

12-10-1991-NA-CT_CHABDPEL_W_CONTRAST-52410 manual: /workspace/dnaxuser-wustl_mmilchenko_adapt/CRLM/DICE_COMPUTE/12-10-1991-NA-CT_CHABDPEL_W_CONTRAST-52410/All_Tumors.nii.gz; shape=(512, 512, 30); voxels=1827
Liver: /data/projects/CRLM/experiments/12-10-1991-NA-CT_CHABDPEL_W_CONTRAST-52410/RESOURCES/Colon-Mets-Liver-Segmentation-Toolbox-20260430/liver_mask.nii.gz; shape=(512, 512, 30)
Manual tumor voxels inside liver: 1825/1827
nnunet: /data/projects/CRLM/experiments/12-10-1991-NA-CT_CHABDPEL_W_CONTRAST-52410/RESOURCES/ABDOMINAL_CT_SEGMENTATION/scan2/nnunet_output/ct_struct.nii.gz; selected label=9; shape=(512, 512, 30)
Native label voxel counts: {0.0: 7187170, 1.0: 43062, 2.0: 47529, 3.0: 2891, 4.0: 113551, 5.0: 17637, 7.0: 180183, 8.0: 271448, 9.0: 849}
Tumor voxels on manual grid: 849; inside liver: 849; intersection: 775
Dice: full image=0.579223, within liver=0.579656
totalseg: /data/projects/CRLM/experiments/12-10-1991-NA-CT_CHABDPEL_W_CONTRAST-52410/RESOURCES/ABDOMINAL_CT_SEGMENT